# 7. Web Scraping — versión para macOS

> **Copia adaptada de `lecture_7R.ipynb`.** El original fue escrito en Windows y no corre en Mac tal cual.
> Cambios aplicados:
>
> | # | Problema en el original | Corrección |
> |---|---|---|
> | 1 | `chromedriver-win64/chromedriver.exe` — ejecutable de Windows | **Selenium Manager**: `webdriver.Chrome()` sin ruta. Descarga solo el driver correcto para tu Chrome, en Mac y en Windows |
> | 2 | El `chromedriver-mac-x64/` de la carpeta es Intel (x86_64) y esta Mac es **arm64** | No se usa; Selenium Manager baja el binario arm64 |
> | 3 | `webdriver.Chrome(executable_path=...)` — **eliminado** en Selenium 4.10 | `webdriver.Chrome(options=...)` |
> | 4 | `find_element_by_xpath`, `find_element_by_id`, `find_elements_by_tag_name`… — **eliminados** en Selenium 4.3 | `find_element(By.XPATH, ...)`, `find_element(By.ID, ...)`, etc. |
> | 5 | Dos celdas con XPaths sueltos como código → `SyntaxError` | Convertidas a celdas de texto |
> | 6 | `to_csv("C:/Users/Alexander/...")` — ruta absoluta de Windows | Rutas relativas con `pathlib` |
> | 7 | `pd.read_html(string)` — deprecado en pandas 2.x | `pd.read_html(StringIO(html))` |
> | 8 | `time.sleep()` fijo — frágil | Esperas explícitas (`WebDriverWait`) |
> | 9 | `dict_scope_options['PERÚ']` — en macOS las tildes vienen en NFD y la comparación falla | Comparación sin tildes con `unidecode` |
> | 10 | `\` en rutas, `"C:\\..."` | `pathlib.Path` (barras correctas en cualquier SO) |

> ⚠️ **Estado del sitio de ONPE (verificado el 18/08/2026):** la web `resultadoshistorico.onpe.gob.pe` carga,
> pero sus archivos de datos (`/assets/json/results/...`) responden **403** detrás de Cloudflare. Es un bloqueo
> del servidor de ONPE, no del código ni del Mac: la propia web muestra la tabla vacía. Las celdas de ONPE
> quedan como material didáctico; al final del notebook hay un respaldo con los datos ya descargados.

## 7.0 Preparación del entorno en Mac

En la terminal, desde la raíz del repo:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install selenium pandas lxml html5lib beautifulsoup4 openpyxl Unidecode ipykernel
python -m ipykernel install --user --name ds-python --display-name "Python (ds-python)"
```

Luego, en Jupyter: **Kernel → Change Kernel → Python (ds-python)**.

No hace falta descargar chromedriver a mano: Selenium ≥ 4.6 lo resuelve solo.

In [ ]:
from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))

# 7. Web Scraping

Web  scraping  is  the  practice  of  gathering  data  through  any  means  otherthan a program interacting with an API (or, obviously, through a human using a webbrowser).  This  is  most  commonly  accomplished  by  writing  an  automated  programthat queries a web server, requests data (usually in the form of the HTML and otherfiles  that  comprise  web  pages),  and  then  parses  that  data  to  extract  needed  information.

# 7.1 Selenium
Selenium automates browsers. That's it! <br>
Selenium is a Python library and tool used for automating web browsers to do a number of tasks. One of such is web-scraping to extract useful data and information that may be otherwise unavailable. <br>
**For this course, we use Chrome.**

## 7.1 Installing Libraries
We need to install these two libraries

In [ ]:
# Instalar (una sola vez). En Mac se recomienda hacerlo dentro de un entorno virtual.
# %pip install selenium pandas lxml html5lib beautifulsoup4 openpyxl Unidecode

# Ya NO hace falta webdriver-manager: Selenium >= 4.6 trae Selenium Manager,
# que descarga solo el chromedriver correcto para tu Chrome y tu arquitectura.

## 7.2 Calling Libraries

In [ ]:
import re
import time
from io import StringIO
from pathlib import Path

import pandas as pd

# Selenium: API vigente (Selenium 4)
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service          # solo si usas un driver manual
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (NoSuchElementException, TimeoutException,
                                        WebDriverException)

import unidecode

# Carpetas de trabajo, relativas al notebook (nada de rutas C:\\Users\\...)
CARPETA = Path.cwd()
IMAGES  = CARPETA / "Images"
OUTPUT  = CARPETA / "output"
IMAGES.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)

print("Carpeta de trabajo:", CARPETA)

## 7.3 Abrir el driver — versión multiplataforma

`abrir_chrome()` reemplaza a todas las variantes de `webdriver.Chrome(executable_path=...)` del notebook original.
Funciona igual en macOS (Intel o Apple Silicon), Windows y Linux.

In [ ]:
def abrir_chrome(headless=False, maximizar=True, espera_carga=45, estrategia="normal"):
    """Abre Chrome con Selenium Manager.

    Selenium >= 4.6 descarga y elige automaticamente el chromedriver que
    corresponde a la version de Chrome instalada y a la arquitectura del
    equipo. Por eso NO se pasa ninguna ruta a chromedriver: el mismo codigo
    corre en Mac (Apple Silicon o Intel), Windows y Linux.

    espera_carga : segundos maximos que driver.get() espera por una pagina.
                   Sin esto, un portal lento deja el notebook colgado.
    estrategia   : "normal" (espera todo), "eager" (solo el DOM) o
                   "none" (devuelve el control de inmediato).
    """
    options = Options()
    options.page_load_strategy = estrategia
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1440,900")
    if headless:
        options.add_argument("--headless=new")
    if maximizar:
        options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(espera_carga)
    driver.set_script_timeout(30)
    driver.implicitly_wait(5)
    return driver


def abrir_url(driver, url, espera_extra=0):
    """driver.get() que no cuelga el notebook.

    Si la pagina supera el page_load_timeout, corta la descarga y sigue
    trabajando con lo que alcanzo a cargar, en vez de bloquearse.
    """
    try:
        driver.get(url)
    except TimeoutException:
        try:
            driver.execute_cdp_cmd("Page.stopLoading", {})   # CDP: no depende del JS de la pagina
        except WebDriverException:
            pass
        print(f"Aviso: {url[:60]}... tardo mas de lo permitido; se sigue con lo cargado.")
    if espera_extra:
        time.sleep(espera_extra)
    return driver.current_url


def esperar(driver, xpath, segundos=30):
    """Espera a que el elemento exista en el DOM y lo devuelve."""
    return WebDriverWait(driver, segundos).until(
        EC.presence_of_element_located((By.XPATH, xpath))
    )


def clic(driver, xpath, segundos=30):
    """Espera a que el elemento sea clickeable, hace scroll hacia el y clickea."""
    el = WebDriverWait(driver, segundos).until(
        EC.element_to_be_clickable((By.XPATH, xpath))
    )
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", el)
    el.click()
    return el


print("Helpers listos: abrir_chrome(), abrir_url(), esperar(), clic()")

### Diagnóstico del entorno (opcional)

In [ ]:
import platform, sys, selenium

print("Sistema      :", platform.system(), platform.release())
print("Arquitectura :", platform.machine())
print("Python       :", sys.version.split()[0])
print("Selenium     :", selenium.__version__)
print("pandas       :", pd.__version__)

_d = abrir_chrome(headless=True, maximizar=False)
print("Chrome       :", _d.capabilities.get("browserVersion"))
print("chromedriver :", _d.capabilities.get("chrome", {}).get("chromedriverVersion", "").split(" ")[0])
_d.quit()

## 7.3 Launch/Set the Driver
This code opens a Chrome Driver. We are going to use it to go navigate on the web.

### Sobre el chromedriver

**Ya no hace falta descargarlo.** Selenium ≥ 4.6 incluye *Selenium Manager*, que detecta tu versión de Chrome y tu arquitectura (arm64 / x86_64) y baja el driver correcto solo.

Si aun así quisieras usar un driver manual desde <https://googlechromelabs.github.io/chrome-for-testing/>, en Mac hay que darle permisos:

```bash
chmod +x chromedriver-mac-arm64/chromedriver
xattr -d com.apple.quarantine chromedriver-mac-arm64/chromedriver
```

y descargar la variante **mac-arm64** (la carpeta `chromedriver-mac-x64` de este repo es Intel y no corresponde a esta Mac).

In [ ]:
# En Windows se usaba `pwd` (magic de IPython). Esto funciona igual en Mac, Linux y Windows:
print(Path.cwd())

In [ ]:
# Abrimos Chrome. Selenium Manager resuelve el driver automaticamente.
driver = abrir_chrome()

url = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
abrir_url(driver, url)

## Chrome is being controlled by automated test software

![Chrome is controlled by automated software](Images/chrome_automated.png)

In [ ]:
# Access to the title
print('Title: ', driver.title)

In [ ]:
# Access to the curent url 
print('Current Page URL: ', driver.current_url)

In [ ]:
# Captura de pantalla (la carpeta Images ya fue creada arriba)
driver.save_screenshot(str(IMAGES / "resultados_presidenciales.png"))

In [ ]:
driver.current_url

In [ ]:
if re.search(r"resultadoshistorico", driver.current_url):
    driver.save_screenshot(str(IMAGES / "resultados_presidenciales.png"))
    print("Resultados Presidenciales saved!")
else:
    print("Page not found")

In [ ]:
#get cookie information
cookies = driver.get_cookies() 
print('Cookies obtained from resultados_presidenciales')
print(cookies)

In [ ]:
# Codigo fuente de la pagina
print(type(driver.page_source))
print(driver.page_source[:1000])

In [ ]:
# ANTES (Windows):
#   service = Service(executable_path="chromedriver-win64/chromedriver.exe")
#   driver  = webdriver.Chrome(service=service)
#
# En Mac ese .exe no existe. Ademas, el binario de chromedriver-mac-x64/ que hay
# en esta carpeta es Intel (x86_64) y esta Mac es Apple Silicon (arm64).
# Solucion portable: dejar que Selenium Manager elija el driver.

driver = abrir_chrome()

# NOTA (verificado el 18/08/2026): larepublica.pe no responde desde esta red
# (curl se queda sin conexion tras 75 s). Usamos un portal que si responde;
# la URL original queda comentada por si el sitio vuelve.
# url = "https://larepublica.pe/politica/2024/01/15/audios-revelan-que-equipo-fiscal-del-caso-cuellos-blancos-armo-plan-contra-harvey-colchado-ministerio-publico-561360"
url = "https://elcomercio.pe/"

abrir_url(driver, url)
print("Cargado:", driver.title)

In [ ]:
print(driver.page_source[:1000])

In [ ]:
driver = abrir_chrome()

url = "https://peru21.pe/lima/peru21-epaper-una-experiencia-sin-limites-noticia/"
abrir_url(driver, url)

In [ ]:
print(driver.page_source[:1000])

In [ ]:
# Refresh the page - 
driver.refresh() #reload or refresh the browser

In [ ]:
driver = abrir_chrome()

driver.maximize_window()

url_1 = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
driver.get( url_1 )


In [ ]:
# driver = abrir_chrome()
# service = Service(executable_path='chromedriver.exe')

driver = abrir_chrome()

driver.maximize_window()

url_1 = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
driver.get( url_1 )

url_2 = "https://larepublica.pe/politica/2024/01/15/audios-revelan-que-equipo-fiscal-del-caso-cuellos-blancos-armo-plan-contra-harvey-colchado-ministerio-publico-561360"
driver.get( url_2 )

driver.back()

In [ ]:
# driver = abrir_chrome()
# service = Service(executable_path='chromedriver.exe')

driver = abrir_chrome()

driver.maximize_window()

url_1 = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
driver.get( url_1 )
time.sleep(3)

url_2 = "https://www.google.com/"
driver.get( url_2 )
time.sleep(3)

driver.back()

In [ ]:
driver = abrir_chrome()

driver.maximize_window()

url_1 = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
driver.get( url_1 )
time.sleep(3)

url_2 = "https://elcomercio.pe/deporte-total/futbol-peruano/noche-blanquiazul-en-vivo-alianza-lima-once-caldas-hoy-dsports-liga-1-max-minuto-a-minuto-desde-el-estadio-nacional-lbposting-noticia/"
driver.get( url_2 )
time.sleep(9)

driver.back()

In [ ]:
driver.close()

In [ ]:
driver.quit()

![Quite and Close](Images/quite_close.png)

In [ ]:
type(driver)

`driver` is an `selenium.webdriver.chrome.webdriver.WebDriver` object. This object has some attributes that will help us to navigate on the web.

Now, you can see in the driver that we are in [this link](https://www.convocatoriascas.com/).

# Extra - Best Practices before working

1. Maximize the browser

In [ ]:
driver = abrir_chrome()

#driver = abrir_chrome()

url = 'https://www.kaspersky.com/resource-center/definitions/cookies'
driver.get( url )

driver.maximize_window()

2. Set the Browser Zoom Level to 100 percent

In [ ]:
driver.execute_script("document.body.style.zoom='100%'")

### 7.4.1. HTML
HTML stands for HyperText Markup Language. You can deduce that it’s a language for creating web pages. It’s not a programming language like Python or Java, but it’s a markup language. It describes the elements of a page through tags characterized by angle brackets.

1. The document always begins and ends using `<html>` and `</html>`.
2. `<body></body>` constitutes the visible part of HTML document.
3. `<h1>` to `<h3>` tags are defined for the headings.

#### 7.4.1.1. HTML Headings
HTML headings are defined with the `<h1>` to `<h6>` tags.
`<h1>` defines the most important heading. `<h6>` defines the least important heading.

We can use text cells since markdown reads html tags.

<h1>This is heading 1</h1>
<h2>This is heading 2</h2>
<h3>This is heading 3</h3>

#### 7.4.1.2. HTML Paragraphs
HTML paragraphs are defined with the `<p>` tag.
`<br>` tag is similar to `"\n"`.

<html>
<br>
<p>My first paragraph.</p> <br>
<p>This is another paragraph for this text cell.</p>
<html>

#### 7.4.1.3. HTML Links
HTML links are defined with the <a> tag:

<a href="http://bayes.cs.ucla.edu/jp_home.html">This is a link for Judea Pearl Website</a>

#### 7.4.1.3. Unordered HTML List
An unordered list starts with the `<ul>` tag. Each list item starts with the `<li>` tag.

<ul>
  <li>Coffee</li>
  <li>Tea</li>
  <li>Milk</li>
</ul>

#### 7.4.1.4. Ordered HTML List
An ordered list starts with the `<ol>` tag. Each list item starts with the `<li>` tag.

<ol>
  <li>Coffee</li>
  <li>Tea</li>
  <li>Milk</li>
</ol>

#### 7.4.1.4. HTML Tables

A table in HTML consists of table cells inside rows and columns. Each table cell is defined by a `<td>` and a `</td>` tag. Each table row starts with a `<tr>` and end with a `</tr>` tag.

<table>
  <tr>
    <th>Manager</th>
    <th>Club</th>
    <th>Nationality</th>
  </tr>
  <tr>
    <td>Mikel Arteta</td>
    <td>Arsenal</td>
    <td>Spain</td>
  </tr>
  <tr>
    <td>Thomas Tuchel</td>
    <td>Chelsea</td>
    <td>Germany</td>
  </tr>
</table>

#### 7.4.1.5. HTML Iframes

An HTML iframe is used to display a web page within a web page.


<!DOCTYPE html>
<html>
  
<head>
    <title>HTML iframe src Attribute</title>
</head>
  
<body style="text-align: center">
    <h1>Diploma</h1>
    <h2>HTML iframe</h2>
    <iframe>
          
        <!DOCTYPE html>
        <html>

        <head>
            <title>New html</title>
        </head>

        <body style="text-align: center">
            <h1>Diploma2</h1>
            <h2>HTML iframe</h2>
            <iframe>

            </iframe>
        </body>

        </html>
    </iframe>
</body>
  
</html>

#### 7.4.1.6. HTML Tags - Key

|Tag|Description|
|---|---|
|`<h1>` to `<h6>`|	Defines HTML headings|
|`<ul>`|	Defines an unordered list|
|`<ol>`|	Defines an ordered list|
|`<p>`|	Defines a paragraph|
|`<a>`|	It is termed as anchor tag and it creates a hyperlink or link.|
|`<div>`|	It defines a division or section within HTML document.|
|`<strong>`|	It is used to define important text.|
|`<table>`|	It is used to present data in tabular form or to create a table within HTML document.|
|`<td>`|	It is used to define cells of an HTML table which contains table data|
|`<iframe>`|	Defines an inline frame|

### 7.4. Identifying elements in a web page

To identify elements of a webpage, we need to inspect the webpage. Open the driver and press `Ctrl`+ `Shift` + `I`.

#### One Element

> ⚠️ Los métodos `find_element_by_*` de esta primera tabla **fueron eliminados en Selenium 4.3**. Se muestran solo como referencia histórica: en la práctica usa siempre la lista de abajo.

|Method|Description|
|---|---|
|find_element_by_id| Use id.|
|find_element_by_name| Use name.|
|find_element_by_xpath| Use Xpath.|
|find_element_by_tag_name| Use HTML tag.|
|find_element_by_class_name| Use class name.|
|find_element_by_css_selector| Use css selector.|

#### One Element - New Documentation

* find_element(By.ID, "id")
* find_element(By.NAME, "name")
* find_element(By.XPATH, "xpath")
* find_element(By.LINK_TEXT, "link text")
* find_element(By.PARTIAL_LINK_TEXT, "partial link text")
* find_element(By.TAG_NAME, "tag name")
* find_element(By.CLASS_NAME, "class name")
* find_element(By.CSS_SELECTOR, "css selector")


#### Multiple  elements

> ⚠️ Igual que arriba: `find_elements_by_*` ya no existe. Usa `find_elements(By.X, ...)`.

|Method|Description|
|---|---|
|find_elements_by_id| Use id.|
|find_elements_by_name| Use name.|
|find_elements_by_xpath| Use Xpath.|
|find_elements_by_tag_name| Use HTML tag.|
|find_elements_by_class_name| Use class name.|
|find_elements_by_css_selector| Use css selector.|

#### Multiple  elements - New Documentation

* find_elements(By.ID, "id")
* find_elements(By.NAME, "name")
* find_elements(By.XPATH, "xpath")
* find_element(By.LINK_TEXT, "link text")
* find_element(By.PARTIAL_LINK_TEXT, "partial link text")
* find_element(By.TAG_NAME, "tag name")
* find_element(By.CLASS_NAME, "class name")
* find_element(By.CSS_SELECTOR, "css selector")


### 7.4.1. Xpath
XPath in Selenium is an XML path used for navigation through the HTML structure of the page. It is a syntax or language for finding any element on a web page using XML path expression.

The basic format of XPath in selenium is explained below with screen shot.
<img src="../_images/x_path.png">

**DO NOT COMPLICATE!**
Finding the XPath of a element:
1. Go to the element
2. Right click
3. Inspect - You may have to do it twice.
4. Go to the selected line
5. Right click
7. Copy 
8. Copy Full Xpath

**Example**

We are going to select `Economistas` option and make a click. Usamos `find_element(By.XPATH, ...)` y `.click()`.

In [ ]:
from selenium.webdriver.common.by import By

In [ ]:
driver = abrir_chrome()

driver.maximize_window()

url_1 = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
driver.get( url_1 )

# resumen_general = driver.find_element( By.XPATH  , '/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]')
# resumen_general.click()
#time.sleep(3)

In [ ]:
resumen_general = driver.find_element( By.XPATH  , '/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]')
resumen_general.click()

In [ ]:
# /html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]/div

**XPaths de referencia**

```
# resumen general
/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]/div
# elecciones presidenciales
/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[2]/div/div/a/div[2]
```

```
/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]/div
/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[2]/div/div/a/div[2]
```

In [ ]:
driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]').click()

In [ ]:
resumen_general = driver.find_element( By.XPATH  , '/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]')
resumen_general.click()

In [ ]:
resumen_general = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]/div')
resumen_general.click()

In [ ]:
#mapdiv > div > div.amcharts-chart-div > svg > g:nth-child(8) > g > g:nth-child(1) > path:nth-child(36)

In [ ]:
resumen_general = driver.find_element(By.XPATH,'/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[1]/img')
resumen_general.click()

In [ ]:
driver.find_element(By.ID, 'select_ambito').click()

In [ ]:
driver.find_element(By.NAME, 'cod_ambito').click()

In [ ]:
# Buscar por ID (API vigente). La forma vieja `find_element_by_id` fue
# eliminada en Selenium 4.3 y ahora lanza AttributeError.
driver.find_element(By.ID, "select_ambito").click()

In [ ]:
# driver.find_element(By.NAME, 'cod_ambito').click()
#driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div/select')

In [ ]:
# driver.find_element(By.CLASS_NAME, 'select_ubigeo')

In [ ]:
# searchBox = driver.find_element(By.ID, 'select_ambito')
# searchBox = driver.find_element(By.XPATH, '//*[@id="select_ambito"]')
# searchBox = driver.find_element(By.CSS_SELECTOR, '#select_ambito')

![Web Element](Images/Web_Elementpng.png)

In [ ]:
searchBox = driver.find_element(By.ID, "select_ambito")
searchBox.get_attribute("value")

In [ ]:
# Las tres formas equivalen a lo mismo
searchBox = driver.find_element(By.ID, "select_ambito")
searchBox = driver.find_element(By.XPATH, '//*[@id="select_ambito"]')
searchBox = driver.find_element(By.CSS_SELECTOR, "#select_ambito")
searchBox.get_attribute("value")

In [ ]:
driver = abrir_chrome()

url = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
abrir_url(driver, url)

In [ ]:
driver.find_element(By.XPATH, "/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[1]/img").click()

In [ ]:
searchBox = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[1]/select/option[2]')
searchBox.click()

In [ ]:
searchBox = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[1]/select/option[2]')
searchBox.text

In [ ]:
searchBox = driver.find_element(By.ID, 'select_ambito')
searchBox

**Suggestion** <br>
We do not recomend to use `tag` at first time since most web pages use nested tags and it is difficult to define a element using HTML tag. However, it is great to find elements that is inside another located element in the web. Let's see the example.

# EXAMPLE USING ONPE WEBPAGE

## Example to extract Table

In [ ]:
driver = abrir_chrome()

url_1 = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
abrir_url(driver, url_1)

In [ ]:
driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]/div').click()

In [ ]:
driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-menu/div/nav/div/div/div[2]/div/div[2]/a').click()

In [ ]:
driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div/select/option[2]').click()

In [ ]:
table_path = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[4]/div[1]/div[3]')

In [ ]:
table_html = table_path.get_attribute("innerHTML")
print(table_html[:500])

In [ ]:
# pandas 2.x pide un buffer, no un string suelto -> StringIO
table = pd.read_html(StringIO(table_html))
table[0]

In [ ]:
driver = abrir_chrome()

url_1 = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
abrir_url(driver, url_1)

# Esperas explicitas en vez de time.sleep fijo: mas rapido y mas confiable
clic(driver, '/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]/div')
clic(driver, '/html/body/onpe-root/onpe-layout-container/onpe-menu/div/nav/div/div/div[2]/div/div[2]/a')
clic(driver, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div[1]/select/option[2]')

table_path = esperar(driver, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[4]/div[1]/div[3]')
table_html = table_path.get_attribute("innerHTML")

table = pd.read_html(StringIO(table_html))
table[0]

In [ ]:
# Clean Table
row_new_columns = table[ 0 ].iloc[ 0 , 2: ]

clean_columns = row_new_columns \
                      .str.replace( " ", "_") \
                      .str.lower().str.replace( "%", "share_") \
                      .apply( lambda x : unidecode.unidecode( x ) ) \
                      .tolist()

# Selecting specific columns
table_clean = table[0].iloc[ 1:, 2: ].copy()

# rename columns
table_clean.columns = clean_columns

In [ ]:
table_clean

In [ ]:
# ANTES: ruta absoluta de Windows ->
#   "C:/Users/Alexander/Documents/GitHub/MediaLab_Summer_Python/Lecture_7/output/..."
# Esa ruta no existe en Mac. Usamos una ruta relativa al notebook:
table_clean.to_csv(OUTPUT / "elecciones_peru_2021.csv", index=False)
print("Guardado en:", OUTPUT / "elecciones_peru_2021.csv")

## [First Round](https://resultadoshistorico.onpe.gob.pe/EG2021/ResumenGeneral/10/T)

In [ ]:
# %pip install lxml Unidecode

In [ ]:
# (los imports ya se hicieron al inicio del notebook)
import numpy as np
import os

# Driver Path Address

In [ ]:
driver = abrir_chrome()

# Extracting all tables

In [ ]:
driver = abrir_chrome()

url_1 = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
abrir_url(driver, url_1)

resumen_general = clic(driver, '/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[1]/img')

In [ ]:
presidential = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[2]/ul/li[1]/a')
presidential.click()

In [ ]:
opt_peru = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[1]/select/option[2]')
opt_peru.click()

## Pesidential results

In [ ]:
# presidential = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-menu/div/nav/div/div/div[2]/div/div[2]/a/span')
# presidential.click

In [ ]:
# # presidential section
# presidential = driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-menu/div/nav/div/div/div[2]/div/div[2]/a" )
# presidential.click()

### Get all elements from all options

In [ ]:
# scope = driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div/select" )
# scope.click()

In [ ]:
#Actualización de las funciones para usar objetos con selenium
regiones = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[2]/select')
regiones

In [ ]:
driver.find_element(By.XPATH, "/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[2]/select/option[2]").text

In [ ]:
#Actualización de las funciones para usar objetos con selenium
regiones = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[2]/select')
regiones.find_elements(By.TAG_NAME, "option")[1].text

In [ ]:
regiones.find_elements(By.TAG_NAME,"option")[1].text

In [ ]:
scope_options = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[1]/select')
scope_options.find_elements(By.TAG_NAME, "option")[2].text

In [ ]:
scope_options = driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[1]/select')

In [ ]:
scope_options.find_elements(By.TAG_NAME, "option")[0].text
scope_options.find_elements(By.TAG_NAME, "option")[1].text
scope_options.find_elements(By.TAG_NAME, "option")[2].text

In [ ]:
scope = driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[1]/select" )
scope

In [ ]:
scope.find_elements(By.TAG_NAME, "option")[2].text

In [ ]:
scope_options = scope.find_elements(By.TAG_NAME, "option")

In [ ]:
scope_options

In [ ]:
dict_scope_options = { i.text : i for i in scope_options }
dict_scope_options

In [ ]:
# There are three options
dict_scope_options.keys()
dict_scope_options

In [ ]:
# OJO (detalle de macOS): las tildes pueden venir descompuestas (NFD),
# de modo que "PERÚ" == "PERU\u0301" falla al comparar con ==.
# Buscamos sin tildes para que funcione igual en Mac y en Windows.
def sin_tildes(txt):
    return unidecode.unidecode(txt).strip().upper()

opcion_peru = next(v for k, v in dict_scope_options.items() if sin_tildes(k) == "PERU")
opcion_peru.click()

We have to be careful since everytime we make a click, the url changes.

### Loop over all departments

In [ ]:
from selenium.webdriver.support.ui import Select  # Import Select class

In [ ]:
# Store all_tables
all_tables = {}

dept_0 = driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[2]/select" )
dept_0

In [ ]:
# All selenium objects in department select
dpt = Select( dept_0 )
#dpt.options[15].text

In [ ]:
dpt.options

In [ ]:
# Get number of total options
num_prov_options = len( dpt.options )
num_prov_options

In [ ]:
# we can loop over all departments
# for dpt_idx in range( num_prov_options ):
# but it will take too much time
# We are going to do it over two departments
for dpt_idx in range( num_prov_options ):
    
    # Get again all departments since HTML is refreshing
    # all elements
    # Click on one specific department
    dpt = Select( driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-rgen-rsgr/div/div[2]/div[1]/div[1]/div/div/div[2]/select" ) )
    department = dpt.options[ dpt_idx ]
    
    # Get departmant name
    dpt_name = department.text
    print(dpt_name)

# Dynamic Pages

In [ ]:
driver = abrir_chrome()

abrir_url(driver, "https://www.legacy.com/obituaries/legacy/obituary-search.aspx?isnew=1&affiliateId=0&stateid=17")

In [ ]:
driver.maximize_window()

In [ ]:
name = driver.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[3]/div/div[1]/input[1]')
name.send_keys("Maria")

In [ ]:
lastname = driver.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[3]/div/div[1]/input[2]')
lastname.send_keys("Brown")

In [ ]:
date_range = driver.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[1]/div[2]/div[1]/select/option[10]')
date_range.click()

In [ ]:
date_begin = driver.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[1]/div[2]/div[3]/div[2]/input')
date_begin.send_keys("01/01/2000")

In [ ]:
date_begin = driver.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[1]/div[2]/div[3]/div[4]/input')
date_begin.send_keys("01/01/2023")

In [ ]:
search = driver.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[5]/div/div[2]/a')
search.click()

In [ ]:
driver = abrir_chrome()

abrir_url(driver, "https://www.legacy.com/obituaries/legacy/obituary-search.aspx?isnew=1&affiliateId=0&stateid=17")

name = esperar(driver, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[3]/div/div[1]/input[1]')
name.send_keys("Maria")

lastname = esperar(driver, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[3]/div/div[1]/input[2]')
lastname.send_keys("Brown")

clic(driver, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[1]/div[2]/div[1]/select/option[10]')

date_begin = esperar(driver, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[1]/div[2]/div[3]/div[2]/input')
date_begin.send_keys("01/01/2000")

date_end = esperar(driver, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[1]/div[2]/div[3]/div[4]/input')
date_end.send_keys("01/01/2023")

clic(driver, '/html/body/div[2]/div[2]/div[2]/form/div[3]/div[1]/div[1]/div/div/div[1]/div[2]/div[5]/div/div[2]/a')

In [ ]:

driver = abrir_chrome()
# Maximize window
driver.maximize_window()
driver.get('https://www.legacy.com/obituaries/legacy/obituary-search.aspx?isnew=1&affiliateId=0&stateid=17')

# range of death
driver.find_element(By.XPATH, '//*[@id="ctl00_ctl00_ContentPlaceHolder1_ContentPlaceHolder1_uxSearchWideControl_ddlSearchRange"]/option[10]').click()


death_begin = driver.find_element(By.XPATH, '//*[@id="ctl00_ctl00_ContentPlaceHolder1_ContentPlaceHolder1_uxSearchWideControl_txtStartDate"]')
death_begin.send_keys('10/10/1994')

death_end = driver.find_element(By.XPATH, '//*[@id="ctl00_ctl00_ContentPlaceHolder1_ContentPlaceHolder1_uxSearchWideControl_txtEndDate"]')    
death_end.send_keys('10/10/2005')

# type the Firstname 
keyword = driver.find_element(By.XPATH, '//*[@id="ctl00_ctl00_ContentPlaceHolder1_ContentPlaceHolder1_uxSearchWideControl_txtFirstName"]')
keyword.send_keys('robert')

# type the Lastname 
keyword = driver.find_element(By.XPATH, '//*[@id="ctl00_ctl00_ContentPlaceHolder1_ContentPlaceHolder1_uxSearchWideControl_txtLastName"]')
keyword.send_keys('brown')

# type the Title 
keyword = driver.find_element(By.XPATH, '//*[@id="ctl00_ctl00_ContentPlaceHolder1_ContentPlaceHolder1_uxSearchWideControl_txtKeyword"]')
keyword.send_keys('professor')

 # Set the state of last residence
driver.find_element(By.XPATH, '//*[@id="ctl00_ctl00_ContentPlaceHolder1_ContentPlaceHolder1_uxSearchWideControl_ddlCountry"]/option[11]').click()
        
# Send information
driver.find_element(By.XPATH, '//*[@id="lnkSearch"]').click()




## Final Example - ONPE

In [ ]:
# Store all_tables
all_tables = {}

driver = abrir_chrome()
# Maximize window
driver.maximize_window()

# go to the link
url_1 = "https://resultadoshistorico.onpe.gob.pe/EG2021/"
driver.get( url_1 )

driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-home-onpe/div[1]/div/div/div/div[2]/div[1]/div/div/a/div[2]/div').click()
time.sleep(2)
driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-menu/div/nav/div/div/div[2]/div/div[2]/a').click()
time.sleep(2)
driver.find_element(By.XPATH, '/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div[1]/select/option[2]').click()
time.sleep(2)

# we can loop over all departments
# for dpt_idx in range( num_prov_options ):
# but it will take too much time
# We are going to do it over two departments
for dpt_idx in range( 2 ):
    
    # Get again all departments since HTML is refreshing
    # all elements
    # Click on one specific department
    dpt = Select( driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div[2]/select" ) )
    department = dpt.options[ dpt_idx ]
    
    # Get departmant name
    dpt_name = department.text
    
    # We select a different department name
    if dpt_name != "--TODOS--" :
        
        # click on department
        department.click()
        
        # Get all elements of province
        prov = Select( driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div[3]/select" ) )
        num_prov_options = len( prov.options )
        
        for prov_idx in range( num_prov_options ):
            
            # Get again all districts since HTML is refreshing
            # all elements
            prov = Select( driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div[3]/select" ) )
            province = prov.options[ prov_idx ]
                
            # Get province name
            prov_name = province.text
            
            if prov_name != "--TODOS--" :
                
                # click on province
                province.click()
                
                # Get all elements from district
                dist = Select( driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div[4]/select" ) )
                num_dist_options = len( dist.options )
                
                for dist_idx in range( num_dist_options ):
                    
                    # Get again all districts since HTML is refreshing
                    # all elements
                    dist = Select( driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[3]/div[1]/div[1]/div/div/div[4]/select" ) )
                    district = dist.options[ dist_idx ]
                    
                    # Get district name
                    dist_name = district.text
                    
                    if dist_name != "-- SELECCIONE --" :
                        
                        # click on district
                        district.click()
                        
                        # Get UBIGEO
                        ubigeo = driver.current_url.split("/")[ -1 ]
                        
                        ## Get table of presidential votes
                        # Get html at this point
                        table_path = driver.find_element(By.XPATH,  "/html/body/onpe-root/onpe-layout-container/onpe-onpe-epres-re/div[1]/div[4]/div[1]/div[3]/div" )
                        table_html = table_path.get_attribute( 'innerHTML' )
                        # Read the table using pandas
                        table = pd.read_html(StringIO(table_html))
                        
                        # Cleaning tables
                        row_new_columns = table[ 0 ].iloc[ 0 , 2: ]
                        clean_columns = row_new_columns \
                                              .str.replace( " ", "_") \
                                              .str.lower().str.replace( "%", "share_") \
                                              .apply( lambda x : unidecode.unidecode( x ) ) \
                                              .tolist()
                        
                        # Selecting specific columns
                        table_clean = table[0].iloc[ 1:, 2: ].copy()
                        
                        # rename columns
                        table_clean.columns = clean_columns
                        
                        # New values to columns 
                        table_clean[ 'department' ] = dpt_name
                        table_clean[ 'province' ]   = prov_name
                        table_clean[ 'district' ]   = dist_name
                        table_clean[ 'ubigeo' ]     = ubigeo
                        
                        # store tables
                        all_tables[ ubigeo ] = table_clean

In [ ]:
final_data = pd.concat( all_tables.values() ).reset_index( drop = True )

In [ ]:
final_data.to_excel(OUTPUT / "example_round.xlsx", index=False)
print("Guardado en:", OUTPUT / "example_round.xlsx")

## Ejemplo Gisella

In [ ]:
driver = abrir_chrome()

url_1 = "https://rpp.pe/noticias/pedro-castillo"
abrir_url(driver, url_1)

# Scroll hasta el final de forma repetida
BOTON_VER_MAS = "/html/body/div[4]/main/div[3]/div[1]/div[2]/button/span"

for _ in range(20):                       # tope de seguridad: evita un bucle infinito
    driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
    time.sleep(3)
    if not driver.find_elements(By.XPATH, BOTON_VER_MAS):
        print("Ya no hay boton 'ver mas': fin del scroll")
        break

In [ ]:
driver.find_element(By.XPATH, "/html/body/div[4]/main/div[3]/div[1]/div[2]/button/span").click()

In [ ]:
driver.find_element(By.XPATH, "/html/body/div[4]/main/div[3]/div[1]/div[2]/button/span").click()

---
## Respaldo: datos ya descargados

Mientras el servidor de ONPE siga devolviendo 403 en sus JSON, la tabla sale vacía.
Para no perder la clase, aquí están los resultados descargados en su momento
(`_data_results/presidential_election_results.csv`), listos para la parte de limpieza y análisis.

In [ ]:
ruta_respaldo = CARPETA / "_data_results" / "presidential_election_results.csv"

if ruta_respaldo.exists():
    respaldo = pd.read_csv(ruta_respaldo)
    print("Filas:", len(respaldo), "| Columnas:", list(respaldo.columns))
    display(respaldo.head())
else:
    print("No se encontro:", ruta_respaldo)

### Cómo comprobar si el sitio de ONPE ya se desbloqueó

Si el JSON responde **200**, el ejemplo original vuelve a funcionar sin tocar nada más.

In [ ]:
import urllib.request, urllib.error

url_json = "https://resultadoshistorico.onpe.gob.pe/assets/json/results/10/000000.json"
try:
    with urllib.request.urlopen(url_json, timeout=20) as r:
        print("Status:", r.status, "-> el sitio ya sirve datos")
except urllib.error.HTTPError as e:
    print("Status:", e.code, "-> sigue bloqueado por el servidor de ONPE (no es tu codigo)")
except Exception as e:
    print("Error de red:", e)